# NAIP XYZ Tile Builder (GDAL-First)

This notebook scans an unzipped NAIP directory, computes inventory and tile estimates, optionally builds XYZ tile pyramids, and creates aggregate VRT mosaics for both visualization modes:

- **RGB** (`bands 1,2,3`)
- **IRG** (`bands 4,1,2`)

The implementation is GDAL-centric (`osgeo.gdal`, `osgeo.osr`, and `gdal2tiles`).

## Cell 2: Configure Paths and Runtime Options

Set your source imagery folders, dry-run behavior, and tiling parameters here.

- `NAIP_ROOTS` accepts **one or more** directories. Each is scanned for TIFF imagery.
- `AGGREGATE_DIR` is where the aggregate VRT mosaics are written; defaults to the first root.
- `DRY_RUN = True`: report counts/plans only, no tile or VRT files written.
- `DRY_RUN = False`: write per-image XYZ tiles and aggregate VRT outputs.

In [1]:
from pathlib import Path
import math
import shutil
import subprocess
import sys
from dataclasses import dataclass
from typing import Iterable

from osgeo import gdal, osr

# --------------------------- User Configuration ---------------------------
# One or more folders containing extracted NAIP image files/folders.
# Add additional Path entries to scan multiple directories.
NAIP_ROOTS = [
    Path('./downloads/castle/naip'),
    # Path('./downloads/creek/naip'),
    # Path('./downloads/czu/naip'),
    # Path('./downloads/northcomplex/naip'),
]

# Raster files to include.
IMAGE_EXTENSIONS = {'.tif', '.tiff'}

# If source imagery lacks embedded CRS, set fallback EPSG code (or None to disable).
FALLBACK_EPSG = 26911

# Tile pyramid settings.
TILE_SIZE = 256
MIN_ZOOM = 0
MAX_ZOOM_CAP = 17

# Runtime controls.
DRY_RUN = True
OVERWRITE_OUTPUT = False

# Output folder names under each image folder.
TILES_DIRNAME = 'tiles'
RGB_DIRNAME = 'rgb'
IRG_DIRNAME = 'irg'

# Band combinations for visualization products.
RGB_BANDS = (1, 2, 3)
IRG_BANDS = (4, 1, 2)

# Aggregate VRT output location. Defaults to a subfolder of the first root.
# Override to any path when processing multiple roots.
AGGREGATE_DIR = NAIP_ROOTS[0] / 'tiles_aggregates'
RGB_AGGREGATE_VRT = AGGREGATE_DIR / 'rgb_all_images.vrt'
IRG_AGGREGATE_VRT = AGGREGATE_DIR / 'irg_all_images.vrt'

print(f'NAIP_ROOTS ({len(NAIP_ROOTS)}):')
for root in NAIP_ROOTS:
    print(f'  {root.resolve()}')
print(f'AGGREGATE_DIR: {AGGREGATE_DIR.resolve()}')
print(f'DRY_RUN: {DRY_RUN}')
print(f'OVERWRITE_OUTPUT: {OVERWRITE_OUTPUT}')

NAIP_ROOTS (1):
  /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip
AGGREGATE_DIR: /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/tiles_aggregates
DRY_RUN: True
OVERWRITE_OUTPUT: False


## Cell 3: Load GDAL Helpers

This cell defines helper functions to:

- find imagery files
- read bounds and resolution with GDAL/OSR
- estimate full-resolution web zoom
- estimate expected XYZ tile counts
- build per-image band VRTs
- call `gdal2tiles` in XYZ mode

In [3]:
@dataclass
class ImagePlan:
    image_path: Path
    image_folder: Path
    rgb_vrt: Path
    irg_vrt: Path
    rgb_tiles_dir: Path
    irg_tiles_dir: Path
    size_bytes: int
    band_count: int
    gsd_m: float | None
    max_zoom: int | None
    tile_count_full_res: int | None
    tile_count_pyramid: int | None


def human_size(num_bytes: int) -> str:
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024
    return f'{num_bytes} B'


def iter_images(root_dir: Path) -> Iterable[Path]:
    for p in root_dir.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            yield p


def make_spatial_ref_from_epsg(epsg_code: int) -> osr.SpatialReference:
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(epsg_code)
    # Ensure consistent lon/lat axis ordering in transformations.
    srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
    return srs


def get_dataset_srs(ds: gdal.Dataset):
    proj_wkt = ds.GetProjection()
    if proj_wkt:
        srs = osr.SpatialReference()
        srs.ImportFromWkt(proj_wkt)
        srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
        return srs

    if FALLBACK_EPSG is not None:
        return make_spatial_ref_from_epsg(FALLBACK_EPSG)

    return None


def pixel_to_geo(gt: tuple[float, float, float, float, float, float], px: float, py: float):
    x = gt[0] + px * gt[1] + py * gt[2]
    y = gt[3] + px * gt[4] + py * gt[5]
    return x, y


def dataset_bounds_wgs84(ds: gdal.Dataset, src_srs: osr.SpatialReference):
    gt = ds.GetGeoTransform(can_return_null=True)
    if gt is None:
        raise ValueError('Dataset has no geotransform; cannot compute extent.')

    w, h = ds.RasterXSize, ds.RasterYSize
    corners_px = [(0, 0), (w, 0), (w, h), (0, h)]
    corners_geo = [pixel_to_geo(gt, px, py) for px, py in corners_px]

    wgs84 = make_spatial_ref_from_epsg(4326)
    tx = osr.CoordinateTransformation(src_srs, wgs84)
    lon_lat = [tx.TransformPoint(x, y)[:2] for x, y in corners_geo]

    lons = [pt[0] for pt in lon_lat]
    lats = [pt[1] for pt in lon_lat]
    return min(lons), min(lats), max(lons), max(lats)


def estimate_gsd_meters(ds: gdal.Dataset, src_srs: osr.SpatialReference):
    gt = ds.GetGeoTransform(can_return_null=True)
    if gt is None:
        return None

    # Compute georeferenced coordinates for the origin and adjacent pixels.
    p0 = pixel_to_geo(gt, 0.0, 0.0)
    px = pixel_to_geo(gt, 1.0, 0.0)
    py = pixel_to_geo(gt, 0.0, 1.0)

    web_merc = make_spatial_ref_from_epsg(3857)
    tx = osr.CoordinateTransformation(src_srs, web_merc)

    p0m = tx.TransformPoint(*p0)
    pxm = tx.TransformPoint(*px)
    pym = tx.TransformPoint(*py)

    x_res = math.hypot(pxm[0] - p0m[0], pxm[1] - p0m[1])
    y_res = math.hypot(pym[0] - p0m[0], pym[1] - p0m[1])
    return float((x_res + y_res) / 2.0)


def zoom_for_full_resolution(gsd_m_per_px: float) -> int:
    if gsd_m_per_px <= 0:
        return MIN_ZOOM
    z = math.ceil(math.log2(156543.03392804097 / gsd_m_per_px))
    return max(MIN_ZOOM, min(MAX_ZOOM_CAP, z))


def lon_to_xtile(lon: float, z: int) -> int:
    n = 2 ** z
    x = int(math.floor((lon + 180.0) / 360.0 * n))
    return max(0, min(n - 1, x))


def lat_to_ytile(lat: float, z: int) -> int:
    lat = max(-85.05112878, min(85.05112878, lat))
    n = 2 ** z
    lat_rad = math.radians(lat)
    y = int(math.floor((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n))
    return max(0, min(n - 1, y))


def tile_count_for_bounds_at_zoom(west: float, south: float, east: float, north: float, z: int) -> int:
    # Keep east slightly inside tile space so edge-aligned extents do not overcount.
    east_adj = min(east, 179.999999)
    x0 = lon_to_xtile(west, z)
    x1 = lon_to_xtile(east_adj, z)
    y0 = lat_to_ytile(north, z)
    y1 = lat_to_ytile(south, z)
    return max(0, x1 - x0 + 1) * max(0, y1 - y0 + 1)


def tile_count_for_pyramid(west: float, south: float, east: float, north: float, z_min: int, z_max: int) -> int:
    total = 0
    for z in range(z_min, z_max + 1):
        total += tile_count_for_bounds_at_zoom(west, south, east, north, z)
    return total


def build_band_vrt(src_image: Path, vrt_path: Path, bands: tuple[int, int, int]):
    vrt_path.parent.mkdir(parents=True, exist_ok=True)
    ds = gdal.Translate(str(vrt_path), str(src_image), format='VRT', bandList=list(bands))
    if ds is None:
        raise RuntimeError(f'Failed to build VRT: {vrt_path}')
    ds = None


def run_gdal2tiles_xyz(src_vrt: Path, dst_dir: Path, z_max: int):
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    if OVERWRITE_OUTPUT and dst_dir.exists():
        shutil.rmtree(dst_dir)

    cmd = [
        sys.executable, '-m', 'osgeo_utils.gdal2tiles',
        '--xyz',
        '--tilesize', str(TILE_SIZE),
        '--resampling', 'bilinear',
        '-z', f'{MIN_ZOOM}-{z_max}',
        '--webviewer', 'none',
        str(src_vrt),
        str(dst_dir),
    ]
    subprocess.run(cmd, check=True)

## Cell 4: Discover Imagery and Build a Tile Plan

This cell scans the NAIP root folder, validates each dataset, computes approximate full-resolution zoom, and estimates tile counts.

Dry-run metrics reported include:

- number of image files discovered
- total source bytes
- expected tiles per image and across the full pyramid (`z=0..z_max`)

In [4]:
gdal.UseExceptions()

# Validate all roots up front before scanning.
missing = [r for r in NAIP_ROOTS if not r.exists()]
if missing:
    raise FileNotFoundError(f'NAIP root(s) not found: {[str(m) for m in missing]}')

plans: list[ImagePlan] = []
total_bytes = 0

# Collect images from all configured roots.
all_images = sorted(img for root in NAIP_ROOTS for img in iter_images(root))

for image_path in all_images:
    size_bytes = image_path.stat().st_size
    total_bytes += size_bytes

    ds = gdal.Open(str(image_path), gdal.GA_ReadOnly)
    if ds is None:
        print(f'[skip] Could not open raster: {image_path}')
        continue

    band_count = ds.RasterCount
    if band_count < 4:
        print(f'[skip] {image_path} has {band_count} band(s); need >= 4 for IRG.')
        ds = None
        continue

    src_srs = get_dataset_srs(ds)
    if src_srs is None:
        print(f'[skip] {image_path} has no CRS and FALLBACK_EPSG is disabled.')
        ds = None
        continue

    try:
        west, south, east, north = dataset_bounds_wgs84(ds, src_srs)
        gsd_m = estimate_gsd_meters(ds, src_srs)
    except Exception as exc:
        print(f'[skip] {image_path} metadata error: {exc}')
        ds = None
        continue

    ds = None

    if gsd_m is None:
        z_max = None
        tiles_full = None
        tiles_pyramid = None
    else:
        z_max = zoom_for_full_resolution(gsd_m)
        tiles_full = tile_count_for_bounds_at_zoom(west, south, east, north, z_max)
        tiles_pyramid = tile_count_for_pyramid(west, south, east, north, MIN_ZOOM, z_max)

    image_folder = image_path.parent
    tiles_root = image_folder / TILES_DIRNAME
    rgb_tiles_dir = tiles_root / RGB_DIRNAME
    irg_tiles_dir = tiles_root / IRG_DIRNAME
    vrt_root = tiles_root / '_vrt'
    rgb_vrt = vrt_root / f'{image_path.stem}_rgb.vrt'
    irg_vrt = vrt_root / f'{image_path.stem}_irg.vrt'

    plans.append(
        ImagePlan(
            image_path=image_path,
            image_folder=image_folder,
            rgb_vrt=rgb_vrt,
            irg_vrt=irg_vrt,
            rgb_tiles_dir=rgb_tiles_dir,
            irg_tiles_dir=irg_tiles_dir,
            size_bytes=size_bytes,
            band_count=band_count,
            gsd_m=gsd_m,
            max_zoom=z_max,
            tile_count_full_res=tiles_full,
            tile_count_pyramid=tiles_pyramid,
        )
    )

print('=== Discovery Summary ===')
print(f'Roots scanned: {len(NAIP_ROOTS)}')
print(f'Images discovered: {len(plans)}')
print(f'Total source size: {human_size(total_bytes)}')

rgb_total_expected = sum(p.tile_count_pyramid or 0 for p in plans)
irg_total_expected = sum(p.tile_count_pyramid or 0 for p in plans)
print(f'Expected RGB tiles across full pyramid: {rgb_total_expected}')
print(f'Expected IRG tiles across full pyramid: {irg_total_expected}')
print(f'Expected total tiles (RGB + IRG): {rgb_total_expected + irg_total_expected}')

for p in plans:
    zoom_text = f'z={p.max_zoom}' if p.max_zoom is not None else 'unavailable'
    gsd_text = f'{p.gsd_m:.3f} m/px' if p.gsd_m is not None else 'unavailable'
    full_text = p.tile_count_full_res if p.tile_count_full_res is not None else 'unavailable'
    pyr_text = p.tile_count_pyramid if p.tile_count_pyramid is not None else 'unavailable'
    print('---')
    print(f'Image: {p.image_path}')
    print(f'Bands: {p.band_count} | Size: {human_size(p.size_bytes)}')
    print(f'Estimated GSD: {gsd_text} | Full-res zoom: {zoom_text}')
    print(f'Estimated tiles at full-res zoom: {full_text}')
    print(f'Estimated tiles in pyramid (z={MIN_ZOOM}..{zoom_text}): {pyr_text}')

=== Discovery Summary ===
Roots scanned: 1
Images discovered: 120
Total source size: 55.97 GB
Expected RGB tiles across full pyramid: 139820
Expected IRG tiles across full pyramid: 139820
Expected total tiles (RGB + IRG): 279640
---
Image: downloads/castle/naip/cogs/postfire/m_3611835_se_11_060_20220707.tif
Bands: 4 | Size: 508.52 MB
Estimated GSD: 0.747 m/px | Full-res zoom: z=17
Estimated tiles at full-res zoom: 864
Estimated tiles in pyramid (z=0..z=17): 1208
---
Image: downloads/castle/naip/cogs/postfire/m_3611835_sw_11_060_20220703.tif
Bands: 4 | Size: 487.32 MB
Estimated GSD: 0.747 m/px | Full-res zoom: z=17
Estimated tiles at full-res zoom: 832
Estimated tiles in pyramid (z=0..z=17): 1187
---
Image: downloads/castle/naip/cogs/postfire/m_3611837_se_11_060_20220704.tif
Bands: 4 | Size: 493.91 MB
Estimated GSD: 0.747 m/px | Full-res zoom: z=17
Estimated tiles at full-res zoom: 864
Estimated tiles in pyramid (z=0..z=17): 1210
---
Image: downloads/castle/naip/cogs/postfire/m_3611837_

Warning 1: m_3611835_se_11_060_20220707.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611835_sw_11_060_20220703.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611837_se_11_060_20220704.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611843_ne_11_060_20220707.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611843_nw_11_060_20220703.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as

## Cell 5: Build Per-Image XYZ Tiles (RGB + IRG)

This cell creates, for each source image folder:

- `tiles/rgb` for bands `1,2,3`
- `tiles/irg` for bands `4,1,2`

When `DRY_RUN=True`, it does not write anything and only reports what would be built.

In [4]:
rgb_vrts: list[Path] = []
irg_vrts: list[Path] = []

rgb_tiles_written = 0
irg_tiles_written = 0
images_tiled = 0
images_skipped = 0

for p in plans:
    if p.max_zoom is None:
        images_skipped += 1
        print(f'[skip] Missing zoom estimate for {p.image_path}')
        continue

    rgb_vrts.append(p.rgb_vrt)
    irg_vrts.append(p.irg_vrt)

    # Expected pyramid count is same for RGB and IRG for the same image extent/zoom.
    expected_count = p.tile_count_pyramid or 0

    if DRY_RUN:
        print(f"[dry-run] {p.image_path.name}")
        print(f'  - mkdir: {p.rgb_tiles_dir}')
        print(f'  - mkdir: {p.irg_tiles_dir}')
        print(f'  - build VRT: {p.rgb_vrt} (bands {RGB_BANDS})')
        print(f'  - build VRT: {p.irg_vrt} (bands {IRG_BANDS})')
        print(f'  - gdal2tiles RGB z={MIN_ZOOM}..{p.max_zoom} -> {p.rgb_tiles_dir}')
        print(f'  - gdal2tiles IRG z={MIN_ZOOM}..{p.max_zoom} -> {p.irg_tiles_dir}')
        print(f'  - estimated RGB tiles: {expected_count}')
        print(f'  - estimated IRG tiles: {expected_count}')
        continue

    # Build image-specific VRTs that map source bands into visualization order.
    build_band_vrt(p.image_path, p.rgb_vrt, RGB_BANDS)
    build_band_vrt(p.image_path, p.irg_vrt, IRG_BANDS)

    # Build standard XYZ folder pyramids using gdal2tiles.
    run_gdal2tiles_xyz(p.rgb_vrt, p.rgb_tiles_dir, p.max_zoom)
    run_gdal2tiles_xyz(p.irg_vrt, p.irg_tiles_dir, p.max_zoom)

    rgb_tiles_written += expected_count
    irg_tiles_written += expected_count
    images_tiled += 1
    print(f'[done] {p.image_path.name} | z_max={p.max_zoom}')

print('=== XYZ Build Summary ===')
if DRY_RUN:
    print('Mode: DRY-RUN (no files written)')
    print(f'Images planned: {len(plans)}')
    print(f'Images skipped: {images_skipped}')
else:
    print('Mode: WRITE')
    print(f'Images tiled: {images_tiled}')
    print(f'Images skipped: {images_skipped}')
    print(f'Approx RGB tiles written: {rgb_tiles_written}')
    print(f'Approx IRG tiles written: {irg_tiles_written}')
    print(f'Approx total tiles written: {rgb_tiles_written + irg_tiles_written}')

[dry-run] m_3611835_se_11_060_20220707.tif
  - mkdir: downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/tiles/rgb
  - mkdir: downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/tiles/irg
  - build VRT: downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/tiles/_vrt/m_3611835_se_11_060_20220707_rgb.vrt (bands (1, 2, 3))
  - build VRT: downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/tiles/_vrt/m_3611835_se_11_060_20220707_irg.vrt (bands (4, 1, 2))
  - gdal2tiles RGB z=0..18 -> downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/tiles/rgb
  - gdal2tiles IRG z=0..18 -> downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/tiles/irg
  - estimated RGB tiles: 4484
  - estimated IRG tiles: 4484
[dry-run] m_3611835_sw_11_060_20220703.tif
  - mkdir: downloads/castle/naip/postfire/m_3611835_sw_11_060_20220703/tiles/rgb
  - mkdir: downloads/castle/naip/postfire/m_3611835_sw_11_060_20220703/tiles/irg
  - build VRT: downloads/castle/naip/postfire/m_

## Cell 6: Build Aggregate VRT Mosaics (RGB + IRG)

This cell creates two aggregate VRT files that mosaic all per-image visualization VRTs:

- `rgb_all_images.vrt`
- `irg_all_images.vrt`

These VRTs provide a single virtual dataset per visualization mode.

In [5]:
def build_aggregate_vrt(vrt_list: list[Path], output_vrt: Path):
    if not vrt_list:
        print(f'[skip] No source VRTs for {output_vrt.name}')
        return

    output_vrt.parent.mkdir(parents=True, exist_ok=True)
    ds = gdal.BuildVRT(
        str(output_vrt),
        [str(p) for p in vrt_list],
        options=gdal.BuildVRTOptions(resolution='highest', allowProjectionDifference=True),
    )
    if ds is None:
        raise RuntimeError(f'Failed to build aggregate VRT: {output_vrt}')
    ds = None
    print(f'[done] {output_vrt}')


if DRY_RUN:
    print('[dry-run] Aggregate VRT targets:')
    print(f'  - {RGB_AGGREGATE_VRT}')
    print(f'  - {IRG_AGGREGATE_VRT}')
    print('Set DRY_RUN=False and run Cell 5 + Cell 6 to write these files.')
else:
    # Keep only VRTs that exist on disk (they are created in Cell 5 during write mode).
    rgb_existing = [p for p in rgb_vrts if p.exists()]
    irg_existing = [p for p in irg_vrts if p.exists()]

    build_aggregate_vrt(rgb_existing, RGB_AGGREGATE_VRT)
    build_aggregate_vrt(irg_existing, IRG_AGGREGATE_VRT)

[dry-run] Aggregate VRT targets:
  - downloads/castle/naip/tiles_aggregates/rgb_all_images.vrt
  - downloads/castle/naip/tiles_aggregates/irg_all_images.vrt
Set DRY_RUN=False and run Cell 5 + Cell 6 to write these files.


## Run Order

1. Run Cell 3 to set configuration (especially `NAIP_ROOT` and `DRY_RUN`).
2. Run Cell 5 to discover imagery and estimate tile counts.
3. Leave `DRY_RUN=True` for planning only, or switch to `False` to write outputs.
4. Run Cell 7 to create per-image `tiles/rgb` and `tiles/irg` XYZ pyramids.
5. Run Cell 9 to build aggregate RGB/IRG VRT mosaics.